## 1주차: Basic

### 과제 1
`temp_samples = [72, 75, 79, 88, 91, 85]` 리스트를 만들고, **85 초과 값만** 출력하세요.

### 과제 2
`is_overheat(temp)` 함수를 작성하고, `temp >= 90`이면 `True`, 아니면 `False`를 반환하게 만드세요.

### 과제 3
NumPy를 사용해 `temp_samples`의 최솟값/최댓값/평균을 계산하세요.

In [ ]:
# 1
temp_samples = [72, 75, 79, 88, 91, 85]
# print([tmp for tmp in temp_samples if tmp > 85])
for temp in temp_samples:
    if temp > 85: print(temp)
# 2
def is_overheat(temp): return temp >= 90

# 3
import numpy as np
nptemp_samples = np.array(temp_samples)
print(nptemp_samples.min(), nptemp_samples.max(), nptemp_samples.mean())


### 미니 미션

아래 조건을 만족하는 코드를 작성해보세요.

- `speed = np.array([35, 42, 51, 49, 62, 71, 68, 55])`
- 속도 50 이상만 추출
- 추출된 값의 평균 계산
- 평균이 60 이상이면 `"고속 주행 구간"` 출력, 아니면 `"일반 주행 구간"` 출력

추가 도전:
- 인덱스까지 함께 출력해 어떤 시점에서 50 이상이었는지 확인

In [ ]:
import numpy as np

speed = np.array([35, 42, 51, 49, 62, 71, 68, 55])
# mask = speed >= 50
# speed[mask]
if speed[speed >= 50].mean() >= 60: print("고속 주행 구간")
else : print("일반 주행 구간")
print("위치:", np.where(speed >= 50)[0]) # np.where는 튜플로 반환하므로 [0]으로 위치 배열을 추출

# idx, = np.where(speed >= 50)
# print("위치:", idx)

## 2주차: IMU

### 필터링 기법
1. **Average Filter**: 전체 데이터의 평균값으로 필터링
2. **Moving Average Filter**: 슬라이딩 윈도우를 사용한 이동 평균
3. **Exponential Moving Average Filter**: 가중 평균을 사용한 지수 이동 평균

In [ ]:

class MotionSensorProcessor:
    """차량 모션 센서 데이터 처리 클래스"""

    def __init__(self):
        self.data = {}

    # 1. 평균 필터
    def average_filter(self, data: np.ndarray) -> np.ndarray: # 변수명은 ndarray임. np.array([])는 전환해주는 함수

        filtered_data = np.zeros_like(data)
        for i in range(len(data)):
            filtered_data[i] = data[:i+1].mean() #numpy 확실하니까 가능.
            # filtered_data[i] = np.mean(data[:i+1])

        return filtered_data
    
    # 2. 이동 평균 필터    
    def mov_avg_filter(self, data:np.ndarray, win_size: int =5) -> np.ndarray:
        if win_size <= 0: raise ValueError("윈도우 크기는 양수여야 합니다.")
        
        filtered_data = np.zeros_like(data, dtype=float) 
        # .mean()이 float을 뱉어도 filtered_data가 float여야 함. 
        # 만약 data가 int면 int로 저장됨
        for i in range(len(data)):
            start_idx = max(0, i + 1 - win_size)
            end_idx = i+1
            filtered_data[i] = data[start_idx:end_idx].mean()
        return filtered_data
    
    # 3. 지수 이동 평균 필터
    def exp_mov_avg_filter(self, data: np.ndarray, alpha: float = 0.3) -> np.ndarray:

        if alpha <= 0 or alpha > 1: raise ValueError("alpha 값은 0과 1 사이여야 합니다.")
        
        filtered_data = np.zeros_like(data, dtype=float)
        filtered_data[0] = data[0]
        
        for i in range(1, len(data)):
            filtered_data[i] = (1 - alpha) * filtered_data[i-1] + alpha * data[i]
        return filtered_data

### 3. Pulse Counting Method

일정 시간 동안의 펄스 개수로 속도 추정

**M-Method**: N = 펄스 개수, ΔT = 고정 시간 간격

### 수식
```
ω = 2π × N / (Z × ΔT)
```
- ω: 각속도 (rad/s)
- N: 샘플링 주기 동안의 펄스 수
- Z: 회전당 펄스 수 (pulses per revolution)
- ΔT: 샘플링 주기 (sec)


In [ ]:
def pulse_counting_method(self, time:np.ndarray, pulses:np.ndarray, pulses_per_revolution: int, counting_time_interval: float) ->np.ndarray:

	est_ang_vel = np.zeros(len(time))

	tmp_pulse_count = 0
	curr_est_ang_vel = 0.0
	prev_time = time[0] # 미리 초기화
	prev_pulse =pulses[0] # 미리 초기화

	for idx in range(1, len(time)):
		if pulses[idx] == 1 and prev_pulse == 0: # 스칼라간의 비교는 'and'
			tmp_pulse_count += 1

		delta_time = time[idx] - prev_time
		
		if delta_time >= counting_time_interval:
			curr_est_ang_vel = tmp_pulse_count * 2 * np.pi / (pulses_per_revolution * counting_time_interval)
			tmp_pulse_count = 0
			prev_time = time[idx]
		 
		est_ang_vel[idx] = curr_est_ang_vel
		prev_pulse = pulses[idx]
		
	return est_ang_vel

### 4. Pulse Timing Method

펄스 간 시간 간격으로 속도 추정

**T-Method**: N = 1, ΔT = 펄스 간 시간 간격  

### 수식
```
ω = 2π / (Z × ΔTd)
```
- ω: 각속도 (rad/s)
- Z: 회전당 펄스 수 (pulses per revolution)
- ΔTd: 펄스 간격


In [ ]:
def pulse_timing_method(self, time: np.ndarray, pulses: np.ndarray, pulses_per_revolution: int) ->np.ndarray:

	prev_time = time[0]
	prev_pulse = pulses[0]
	curr_est_ang_vel = 0.0
	est_ang_vel = np.zeros(len(time))

	for idx in range(1,len(time)):
		if pulses[idx] == 1 and prev_pulse == 0: 
			delta_time = time[idx] - prev_time

			if delta_time < 1e-3:
				delta_time = 1e-3

			curr_est_ang_vel = 2 * np.pi / (pulses_per_revolution * delta_time)
			prev_time = time[idx]
	
		est_ang_vel[idx] = curr_est_ang_vel
		prev_pulse = pulses[idx]

	return est_ang_vel

### 5. DR(Dead Reckoning)을 통한 위치 추정

Dead Reckoning(DR): **IMU 센서(자이로의 yaw rate)와 속도 정보(Wheel Odometry를 통해 취득)를 활용하여 차량의 위치를 적분적으로 추정하는 방법**
GPS/INS 데이터로부터 얻은 초기 위치와 속도를 바탕으로, IMU yaw rate와 속도를 적분하여 DR 경로를 계산

**장점**: GPS 없이도 연속적으로 위치 추정 가능

**단점**: 센서 노이즈와 바이어스로 인해 시간이 지남에 따라 누적 오차 발생

### DR 위치 추정 과정
1. **센서 데이터 동기화**  
   - IMU 시간축 기준으로 Wheel Odometry 속도 데이터를 보간(interpolation)  
   - `np.interp()`를 사용하여 imu_time 기준 속도 배열 생성  

2. **초기 상태 설정**  
   - 시작 위도, 경도, 방위각(azimuth) 추출  
   - 지구 반경 (R = 6378137m) 사용하여 위도/경도 변화량 계산 준비  

3. **방향(Heading) 업데이트**  
   - IMU yaw rate $(\omega)$ 적분  
   
   $\theta_k = \theta_{k-1} + \omega \cdot \Delta t$  

4. **변위 계산**  
   - 선속도 ($v$) 와 방향 $(\theta)$ 를 이용한 이동 거리 계산  
   
   $dx = v \cdot \cos(\theta) \cdot \Delta t$  
   
   $dy = v \cdot \sin(\theta) \cdot \Delta t$  

5. **위도/경도 업데이트**  
   - 지구 반경을 고려한 위도, 경도 증분 계산  
   
   $\Delta \phi = \frac{dy}{R}, \quad$
   $\Delta \lambda = \frac{dx}{R \cdot \cos(\phi)}$
     
   - 최종 위치 누적

   $\phi_k = \phi_{k-1} + \Delta \phi, \quad$
   $\lambda_k = \lambda_{k-1} + \Delta \lambda$


In [ ]:
def calc_dr_traj(self, imu_data: dict, inspvax_data: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    imu_time = np.array(imu_data['time'])
    imu_yaw_rate = np.array([d.angular_velocity.z for d in imu_data['data']])
    
    inspvax_time = np.array(inspvax_data['time'])
    north_vel    = np.array([d.north_velocity for d in inspvax_data['data']])
    east_vel     = np.array([d.east_velocity for d in inspvax_data['data']])
    
    start_time = imu_time[0]
    imu_time -= start_time
    inspvax_time -= start_time
    
    velocity_interp = np.interp(imu_time, inspvax_time, np.sqrt(north_vel**2 + east_vel**2))
    current_vel = 0.0
    
    R = 6378137.0
    
    d0 = inspvax_data['data'][0]
    lat_rad = np.deg2rad(d0.latitude)
    lon_rad = np.deg2rad(d0.longitude)
    theta   = np.deg2rad(90 - d0.azimuth)
    
    N = len(imu_time)
    dr_lat = np.zeros(N, float)
    dr_lon = np.zeros(N, float)
    
    dr_lat[0] = np.rad2deg(lat_rad)
    dr_lon[0] = np.rad2deg(lon_rad)
    
    for i in range(1,N):
        dt = imu_time[i] - imu_time[i-1]
        theta += imu_yaw_rate[i] * dt
        current_vel = velocity_interp[i]
        dx = current_vel * np.cos(theta) * dt
        dy = current_vel * np.sin(theta) * dt
        
        lat_rad += dy / R
        lon_rad += dx / (R * np.cos(lat_rad))
        
        dr_lat[i] = np.rad2deg(lat_rad)
        dr_lon[i] = np.rad2deg(lon_rad)
        
    return dr_lat, dr_lon, imu_time

## 3주차:GNSS

### 1. NMEA 데이터 분석

NMEA (National Marine Electronics Association) 0183은 GPS 수신기에서 가장 널리 사용되는 표준 통신 프로토콜입니다. 
이번 실습에서는 GGA와 RMC 메시지를 중심으로 NMEA 데이터를 분석합니다.

### NMEA 메시지 종류
- **GGA (Global Positioning System Fix Data)**: 위치, 고도, HDOP, 위성 수 등
- **RMC (Recommended Minimum)**: 위치, 속도, 방향, 날짜, 상태 등

In [ ]:
    def _parse_gga(self, line: str) -> Optional[Dict]:
        """
        GGA 메세지 구조: $GPGGA,<time>,<latitude>,<latitude_hemisphere>,<longitude>,<longitude_hemisphere>,<fix_quality>,<num_satellites>,<hdop>,<altitude>,<altitude_unit>,<geoid_height>,<checksum>
        """
        try:
            parts = line.split(',')
            if len(parts) < 15:
                return None
                
            # 위도 변환
            lat_str = parts[2] # 위도 문자열 추출 - parts 사용
            lat_hemisphere = parts[3] # 위도 방향 추출
            if lat_str and lat_hemisphere:
                lat_deg = int(lat_str[:2])
                lat_min = float(lat_str[2:])
                latitude = lat_deg + lat_min / 60.0
                if lat_hemisphere == 'S':
                    latitude = -latitude
            else:
                latitude = None
                
            # 경도 변환
            lon_str = parts[4] # 경도 문자열 추출
            lon_hemisphere = parts[5] # 경도 방향 추출
            if lon_str and lon_hemisphere:
                lon_deg = int(lon_str[:3])
                lon_min = float(lon_str[3:])
                longitude = lon_deg + lon_min / 60.0
                if lon_hemisphere == 'W':
                    longitude = -longitude
            else:
                longitude = None
                
            return {
                'time': parts[1], # UTC 시간 추출
                'latitude': latitude,
                'longitude': longitude,
                'fix_quality': int(parts[6]) if parts[6] else 0, # FIX 품질 추출
                'num_satellites': int(parts[7]) if parts[7] else 0, # 위성 수 추출
                'hdop': float(parts[8]) if parts[8] else 0.0, # HDOP 추출
                'altitude': float(parts[9]) if parts[9] else 0.0, # 고도 추출
                'altitude_unit': parts[10] if parts[10] else '', # 고도 단위 추출
                'geoid_height': float(parts[11]) if parts[11] else 0.0 # 지구 고도 추출 (undulation)
            }
        except (ValueError, IndexError):
            return None
    
    def _parse_rmc(self, line: str) -> Optional[Dict]:
        """
        RMC 메세지 구조: $GPRMC,<time>,<status>,<latitude>,<latitude_hemisphere>,<longitude>,<longitude_hemisphere>,<speed>,<course>
        ,<date>,<magnetic_variation>,<magnetic_variation_hemisphere>,<mode>,<checksum>
        """
        try:
            parts = line.split(',')
            if len(parts) < 12:
                return None
                
            # 위도 변환
            lat_str = parts[3] # 위도 문자열 추출
            lat_hemisphere = parts[4] # 위도 방향 추출
            if lat_str and lat_hemisphere:
                lat_deg = int(lat_str[:2])
                lat_min = float(lat_str[2:])
                latitude = lat_deg + lat_min / 60.0
                if lat_hemisphere == 'S':
                    latitude = -latitude
            else:
                latitude = None
                
            # 경도 변환
            lon_str = parts[5] # 경도 문자열 추출
            lon_hemisphere = parts[6] # 경도 방향 추출
            if lon_str and lon_hemisphere:
                lon_deg = int(lon_str[:3])
                lon_min = float(lon_str[3:])
                longitude = lon_deg + lon_min / 60.0
                if lon_hemisphere == 'W':
                    longitude = -longitude
            else:
                longitude = None
                
            return {
                'time': parts[1], # UTC 시간 추출
                'status': parts[2], # 상태 추출
                'latitude': latitude,
                'longitude': longitude,
                'speed_knots': float(parts[7]) if parts[7] else 0.0, # 속도 추출
                'course': float(parts[8]) if parts[8] else 0.0, # 방향 추출 (Track true)
                'date': parts[9] # 날짜 추출
            }
        except (ValueError, IndexError):
            return None

### 4. 좌표 변환 실습 (WGS84 ↔ ENU)

전역 좌표계(WGS84)와 지역 좌표계(ENU) 간 변환

### 좌표계 설명
- **WGS84**: 세계 측지계, GPS가 사용하는 전역 좌표계 (위도, 경도, 고도)
- **ENU**: East-North-Up, 지역 직교 좌표계 (동쪽, 북쪽, 위쪽)

### 변환 과정
1. **WGS84 → ENU**: 글로벌 좌표를 로컬 좌표로 변환
2. **ENU → WGS84**: 로컬 좌표를 다시 글로벌 좌표로 역변환
3. **정확도 검증**: 원본과 재변환 결과 비교

### 변환 수식
**WGS84 → ENU:**
```
E = Δlon × (N + h) × cos(φ₀)
N = Δlat × (M + h)  
U = Δh
```

**ENU → WGS84:**
```
Δlat = N / (M + h)
Δlon = E / ((N + h) × cos(φ₀))
Δh = U
```

여기서 M은 자오선 곡률반지름, N은 수직 곡률반지름입니다.

In [ ]:
import numpy as np
def wgs84_to_enu(self, llh: np.ndarray, ref_llh: np.ndarray) -> np.ndarray:
    enu = np.zeros_like(llh, float)
    
    ref_lat = np.deg2rad(ref_llh[0])
    ref_lon = np.deg2rad(ref_llh[1])
    ref_height  = ref_llh[2]
    
    meridional_r = self.meridional_radius(ref_llh[0])
    normal_r    = self.normal_radius(ref_llh[0])
    
    lat = np.deg2rad(llh[:, 0])
    lon = np.deg2rad(llh[:, 1])
    height = (llh[:,2])
    
    delta_lat_rad = lat - ref_lat
    delta_lon_rad = lon - ref_lon
    delta_height  = height - ref_height
    
    enu[:, 0] = delta_lon_rad * (normal_r + ref_height) * np.cos(ref_lat)
    enu[:, 1] = delta_lat_rad * (meridional_r + ref_height)
    enu[:, 2] = delta_height
    
    return enu
def enu_to_wgs84(self, enu: np.ndarray, ref_llh: np.ndarray) -> np.ndarray:
    llh = np.zeros_like(enu, float)
    
    ref_lat = np.deg2rad(ref_llh[0])
    ref_lon = np.deg2rad(ref_llh[1])
    ref_height = ref_llh[2]
    
    meridional_r = self.meridional_radius(ref_llh[0])
    normal_r = self.normal_radius(ref_llh[0])
    
    delta_lon_rad = enu[:, 0] / ((normal_r + ref_height) * np.cos(ref_lat))
    delta_lat_rad = enu[:, 1] / (meridional_r + ref_height)
    delta_height = enu[:, 2]
    
    llh[:, 0] = np.rad2deg(ref_lat + delta_lat_rad)
    llh[:, 1] = np.rad2deg(ref_lon + delta_lon_rad)
    llh[:, 2] = ref_height + delta_height
    
    return llh

## 4주차: LiDAR 

LiDAR 파트는 `custom_lidar.py`의 TODO 함수들을 흐름대로 정리하면 크게 다음 4단계로 볼 수 있습니다.

- **ROI Extraction**: 관심 있는 공간 범위만 남기기
- **k-NN Search**: 쿼리 점과 가까운 이웃 찾기
- **Least Squares Fitting**: 직선과 구를 최소 자승법으로 추정하기
- **RANSAC Helpers**: 샘플링, 모델 추정, 거리 계산, 인라이어 분류


### 1. ROI(Region of Interest) 추출

포인트 클라우드 전체를 다 쓰지 않고, 필요한 범위만 남기는 단계입니다.

핵심 아이디어는 축별 조건 마스크를 만든 뒤, 이를 AND 연산으로 합치는 것입니다.

- `x_mask`: X축 범위 필터
- `y_mask`: Y축 범위 필터
- `z_mask`: Z축 범위 필터
- `roi_mask = x_mask & y_mask & z_mask`


In [ ]:
import open3d as o3d
import numpy as np
from typing import Tuple

def custom_extract_roi(
    pcd: o3d.geometry.PointCloud,
    x_range: Tuple[float, float] = (-20, 20),
    y_range: Tuple[float, float] = (-20, 20),
    z_range: Tuple[float, float] = (-3, 3),
) -> Tuple[o3d.geometry.PointCloud, o3d.geometry.PointCloud]:
    
    points = np.array(pcd.points)
    x_mask = (points[:, 0] >= x_range[0]) & (points[:, 0] <= x_range[1])
    y_mask = (points[:, 1] >= y_range[0]) & (points[:, 1] <= y_range[1])
    z_mask = (points[:, 2] >= z_range[0]) & (points[:, 2] <= z_range[1])
    
    roi_mask = x_mask & y_mask & z_mask
    roi_points = points[roi_mask]
    roi_outliers = points[~roi_mask]
    
    roi_pcd = o3d.geometry.PointCloud()
    roi_pcd.points = o3d.utility.Vector3dVector(roi_points)
    
    roi_outliers_pcd = o3d.geometry.PointCloud()
    roi_outliers_pcd.points = o3d.utility.Vector3dVector(roi_outliers)
    
    if len(pcd.colors) > 0:
        colors = np.array(pcd.colors)
        roi_colors = colors[roi_mask]
        roi_pcd.colors = o3d.utility.Vector3dVector(roi_colors)
        
        roi_outliers_colors = colors[~roi_mask]
        roi_outliers_pcd.colors = o3d.utility.Vector3dVector(roi_outliers_colors)
        
    return roi_pcd, roi_outliers_pcd
        
        
    
    

### 2. Brute-force k-NN 검색

KD-Tree를 쓰기 전 가장 단순한 최근접 이웃 탐색입니다.

과정은 다음과 같습니다.

1. 모든 점과 쿼리 점 사이 거리 계산
2. 거리 기준 정렬
3. 앞에서부터 `k`개 선택


In [ ]:
def custom_brute_force_search(
    pcd: o3d.geometry.PointCloud,
    query: np.ndarray,
    k: int,
) -> Tuple[np.ndarray, np.ndarray]:
    
    points = np.array(pcd.points)
    query = np.array(query)
    
    distances = np.linalg.norm(points - query, axis = 1)
    sorted_indices = np.argsort(distances)
    
    k_indices = sorted_indices[:k]
    k_distances = distances[k_indices]
    return k_indices, k_distances

### 3. Least Squares 직선 피팅

직선 방정식 `y = ax + b`를 최소 자승법으로 추정합니다.

행렬 형태로 쓰면 `H @ params = b` 이고, 정규 방정식은 아래와 같습니다.

```
params = (H^T H)^(-1) H^T b
```

여기서 `params = [a, b]^T` 입니다.


In [ ]:
def custom_least_squares_line_fitting(pcd: o3d.geometry.PointCloud) -> np.ndarray:
    points = np.array(pcd.points)
    x, y = points[:, 0], points[:, 1]
    n = len(points)

    H = np.column_stack([x, np.ones(n)])
    b = y

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    return inv_HtH @ Ht_b


### 4. Least Squares 구 피팅

구 방정식은 비선형이지만, 전개 후 선형 형태로 바꿔 최소 자승법을 적용할 수 있습니다.

- 중심: `(cx, cy, cz)`
- 반지름: `r`
- 마지막에 `radius_squared = cx^2 + cy^2 + cz^2 + d` 로 반지름 복원


In [ ]:
def custom_least_squares_sphere_fitting(
    pcd: o3d.geometry.PointCloud,
) -> Tuple[np.ndarray, float]:
    points = np.array(pcd.points)
    x, y, z = points[:, 0], points[:, 1], points[:, 2]
    n = len(points)

    H = np.column_stack([x, y, z, np.ones(n)])
    b = x ** 2 + y ** 2 + z ** 2

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    params = inv_HtH @ Ht_b
    cx, cy, cz = params[0] / 2, params[1] / 2, params[2] / 2
    d = params[3]

    radius_squared = cx ** 2 + cy ** 2 + cz ** 2 + d
    radius = np.sqrt(max(0, radius_squared))
    center = np.array([cx, cy, cz], dtype=float)
    return center, radius


### 5. RANSAC용 샘플링과 모델 추정

RANSAC은 매 반복마다 일부 점만 뽑아서 임시 모델을 만들고, 그 모델이 얼마나 많은 인라이어를 가지는지 평가합니다.

여기서는 다음 보조 함수들이 필요합니다.

- `custom_ransac_sample_points`: 중복 없이 랜덤 샘플링
- `custom_fit_line_from_samples`: 샘플 점들로 직선 추정
- `custom_fit_sphere_from_samples`: 샘플 점들로 구 추정


In [ ]:
def custom_ransac_sample_points(
    pcd: o3d.geometry.PointCloud,
    n_samples: int,
) -> Tuple[np.ndarray, np.ndarray]:
    points = np.array(pcd.points)
    n_points = len(points)

    sampled_indices = np.random.choice(n_points, size=n_samples, replace=False)
    sampled_points = points[sampled_indices]
    return sampled_points, sampled_indices


def custom_fit_line_from_samples(sampled_points: np.ndarray) -> np.ndarray:
    if len(sampled_points) < 2:
        raise ValueError("직선 피팅을 위해서는 최소 2개의 포인트가 필요합니다.")

    x, y = sampled_points[:, 0], sampled_points[:, 1]
    n = len(sampled_points)

    H = np.column_stack([x, np.ones(n)])
    b = y

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    return inv_HtH @ Ht_b


def custom_fit_sphere_from_samples(
    sampled_points: np.ndarray,
) -> Tuple[np.ndarray, float]:
    if len(sampled_points) < 4:
        raise ValueError("구 피팅을 위해서는 최소 4개의 포인트가 필요합니다.")

    x, y, z = sampled_points[:, 0], sampled_points[:, 1], sampled_points[:, 2]
    n = len(sampled_points)

    H = np.column_stack([x, y, z, np.ones(n)])
    b = x ** 2 + y ** 2 + z ** 2

    HtH = H.T @ H
    Ht_b = H.T @ b

    try:
        inv_HtH = np.linalg.inv(HtH)
    except np.linalg.LinAlgError:
        inv_HtH = np.linalg.pinv(HtH)

    params = inv_HtH @ Ht_b
    cx, cy, cz = params[0] / 2, params[1] / 2, params[2] / 2
    d = params[3]

    radius_squared = cx ** 2 + cy ** 2 + cz ** 2 + d
    radius = np.sqrt(max(0, radius_squared))
    center = np.array([cx, cy, cz], dtype=float)
    return center, radius

### 6. 거리 계산과 Inlier / Outlier 분류

RANSAC에서 모델을 하나 세웠다면, 이제 모든 점이 그 모델에 얼마나 잘 맞는지 거리로 평가합니다.

- 직선 거리: `|ax - y + b| / sqrt(a^2 + 1)`
- 구 거리: `|distance_to_center - radius|`
- threshold보다 작으면 inlier, 크거나 같으면 outlier


In [ ]:
def custom_compute_line_distances(
    pcd: o3d.geometry.PointCloud,
    line_params: np.ndarray,
) -> np.ndarray:
    points = np.array(pcd.points)
    x, y = points[:, 0], points[:, 1]
    a, b = line_params

    numerator = np.abs(a * x - y + b)
    denominator = np.sqrt(a ** 2 + 1)
    return numerator / denominator


def custom_compute_sphere_distances(
    pcd: o3d.geometry.PointCloud,
    center: np.ndarray,
    radius: float,
) -> np.ndarray:
    points = np.array(pcd.points)
    distances_to_center = np.linalg.norm(points - center, axis=1)
    return np.abs(distances_to_center - radius)


def custom_find_inliers_outliers(
    distances: np.ndarray,
    threshold: float,
) -> Tuple[np.ndarray, np.ndarray]:
    inlier_indices = np.where(distances < threshold)[0]
    outlier_indices = np.where(distances >= threshold)[0]
    return inlier_indices, outlier_indices


## 5주차: Radar

Radar 파트는 `custom_radar.py`의 TODO 함수들을 흐름대로 정리하면 크게 다음 4단계로 볼 수 있습니다.

- **Noise Filtering**: Statistical Outlier Removal — KD-Tree 기반 K-NN으로 아웃라이어 제거
- **Translation**: 포인트 클라우드 평행 이동
- **Rotation**: Roll·Pitch·Yaw 회전 행렬 생성 및 적용
- **Transformation Matrix**: 4×4 행렬로 R과 t를 통합해 변환
- **DBSCAN Clustering**: 밀도 기반 군집화 직접 구현

### 1. Statistical Outlier Removal (SOR)

각 포인트의 **k-최근접 이웃까지 평균 거리**를 계산하고, 전체 분포 기준 통계적으로 벗어나는 점을 아웃라이어로 제거합니다.

1. KD-Tree로 각 점의 k+1개 이웃 탐색 (자기 자신 포함) → 자기 자신 제외 후 평균 거리 계산
2. 전역 평균 μ, 표준편차 σ 계산
3. `mean_dist > μ + std_ratio×σ` 인 점 = 아웃라이어
4. 인라이어만 남겨 반환

In [ ]:
# class StatisticalOutlierRemoval:
#     def __init__(self, nb_neighbors: int = 20, std_ratio: float = 2.0):
#         self.nb_neighbors = nb_neighbors
#         self.std_ratio = std_ratio
#         self.kdtree_root_ = None
#         self.mean_distances_ = None
#         self.global_mean_ = None
#         self.global_std_ = None
#         self.threshold_lower_ = None
#         self.threshold_upper_ = None
#         self.processing_stats_ = {}

#     def compute_mean_distances(self, points: np.ndarray) -> np.ndarray:
#         pcd = o3d.geometry.PointCloud()
#         pcd.points = o3d.utility.Vector3dVector(points)
#         self.kdtree_root_ = o3d.geometry.KDTreeFlann(pcd)

#         mean_distances = np.zeros(len(points))
#         for i, query_point in enumerate(points):
#             # k+1개 이웃 탐색 (자기 자신 포함)
#             [_, neighbor_indices, squared_distances] = self.kdtree_root_.search_knn_vector_3d(
#                 query_point, self.nb_neighbors + 1
#             )
#             if len(neighbor_indices) > 1:
#                 # squared_distances에서 자기 자신(인덱스 0) 제외
#                 actual_neighbor_distances = squared_distances[1:]
#                 neighbor_distances = np.sqrt(actual_neighbor_distances)
#                 mean_distances[i] = np.mean(neighbor_distances)
#             else:
#                 mean_distances[i] = 0.0
#         return mean_distances

#     def compute_statistics(self, mean_distances: np.ndarray) -> Tuple[float, float, float, float]:
#         global_mean = np.mean(mean_distances)
#         global_std  = np.std(mean_distances)
#         # std_ratio를 사용하여 상·하한 임계값 계산
#         threshold_lower = global_mean - self.std_ratio * global_std
#         threshold_upper = global_mean + self.std_ratio * global_std
#         return global_mean, global_std, threshold_lower, threshold_upper

#     def find_outliers(self, mean_distances: np.ndarray,
#                       threshold_lower: float, threshold_upper: float) -> np.ndarray:
#         # 상한 임계값을 초과하는 점들이 아웃라이어
#         outlier_mask = mean_distances > threshold_upper
#         return outlier_mask

#     def filter_points_sor(self, points: np.ndarray, return_outliers: bool = False):
#         self.mean_distances_ = self.compute_mean_distances(points)
#         self.global_mean_, self.global_std_, self.threshold_lower_, self.threshold_upper_ = \
#             self.compute_statistics(self.mean_distances_)

#         outlier_mask = self.find_outliers(self.mean_distances_, self.threshold_lower_, self.threshold_upper_)
#         inlier_mask = ~outlier_mask
#         # inlier_mask를 사용하여 필터링된 포인트 선택
#         filtered_points = points[inlier_mask]

#         if return_outliers:
#             outlier_points = points[outlier_mask]
#             return filtered_points, outlier_points
#         else:
#             return filtered_points, np.where(outlier_mask)[0]


class StatisticalOutlierRemoval:
    def __init__(self, nb_neighbors: int = 20, std_ratio: float = 2.0):
        self.nb_neighbors = nb_neighbors
        self.std_ratio = std_ratio
        self.kdtree_root_ = None
        self.mean_distances_ = None
        self.global_mean_ = None
        self.global_std_ = None
        self.threshold_upper_ = None

    def compute_mean_distances(self, points: np.ndarray) -> np.ndarray:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points)
        self.kdtree_root_ = o3d.geometry.KDTreeFlann(pcd)

        mean_distances = np.zeros(len(points))
        for i, query_point in enumerate(points):
            [_, _, squared_distances] = self.kdtree_root_.search_knn_vector_3d(
                query_point, self.nb_neighbors + 1
            )
            if len(squared_distances) > 1:
                mean_distances[i] = np.mean(np.sqrt(squared_distances[1:]))
        return mean_distances

    def compute_statistics(self, mean_distances: np.ndarray) -> Tuple[float, float, float]:
        global_mean = np.mean(mean_distances)
        global_std  = np.std(mean_distances)
        threshold_upper = global_mean + self.std_ratio * global_std
        return global_mean, global_std, threshold_upper

    def filter_points_sor(self, points: np.ndarray, return_outliers: bool = False):
        self.mean_distances_ = self.compute_mean_distances(points)
        self.global_mean_, self.global_std_, self.threshold_upper_ = \
            self.compute_statistics(self.mean_distances_)

        outlier_mask = self.mean_distances_ > self.threshold_upper_
        inlier_mask  = ~outlier_mask
        filtered_points = points[inlier_mask]

        if return_outliers:
            return filtered_points, points[outlier_mask]
        else:
            return filtered_points, np.where(outlier_mask)[0]

### 2. Translation (평행 이동)

포인트 클라우드를 각 좌표축 방향으로 평행이동합니다.

$$p' = p + T, \quad T = [t_x,\, t_y,\, t_z]^T$$

In [ ]:
def custom_translation(pcd: o3d.geometry.PointCloud, vector: np.ndarray) -> o3d.geometry.PointCloud:
    xyz = np.asarray(pcd.points)
    # xyz에 vector를 더하여 이동된 좌표 계산 (vector shape=(3,) 이므로 바로 더할 수 있음)
    translated_xyz = xyz + vector

    pcd_translation = o3d.geometry.PointCloud()
    pcd_translation.points = o3d.utility.Vector3dVector(translated_xyz)
    return pcd_translation

### 3. Rotation (회전)

x축 회전 Roll($\phi$), y축 회전 Pitch($\theta$), z축 회전 Yaw($\psi$) 행렬을 순서대로 곱해 최종 회전 행렬을 구합니다.

$$R = R_x(\phi)\,R_y(\theta)\,R_z(\psi)$$

$$R_x = \begin{bmatrix}1&0&0\\0&\cos\phi&-\sin\phi\\0&\sin\phi&\cos\phi\end{bmatrix},\quad
R_y = \begin{bmatrix}\cos\theta&0&\sin\theta\\0&1&0\\-\sin\theta&0&\cos\theta\end{bmatrix},\quad
R_z = \begin{bmatrix}\cos\psi&-\sin\psi&0\\\sin\psi&\cos\psi&0\\0&0&1\end{bmatrix}$$

In [ ]:
import numpy as np
import open3d as o3d

def custom_get_rotation_matrix(angle: np.ndarray) -> np.ndarray:
    """Get rotation matrix from angle [roll, pitch, yaw] (degree)"""
    roll  = np.deg2rad(angle[0])
    pitch = np.deg2rad(angle[1])
    yaw   = np.deg2rad(angle[2])

    # roll 회전 행렬 (x축)
    rotation_matrix_roll = np.array([
        [1, 0,             0            ],
        [0, np.cos(roll), -np.sin(roll) ],
        [0, np.sin(roll),  np.cos(roll) ]
    ])
    # pitch 회전 행렬 (y축)
    rotation_matrix_pitch = np.array([
        [ np.cos(pitch), 0, np.sin(pitch)],
        [0,              1, 0            ],
        [-np.sin(pitch), 0, np.cos(pitch)]
    ])
    # yaw 회전 행렬 (z축)
    rotation_matrix_yaw = np.array([
        [np.cos(yaw), -np.sin(yaw), 0],
        [np.sin(yaw),  np.cos(yaw), 0],
        [0,            0,           1]
    ])

    # roll, pitch, yaw 회전 행렬을 곱하여 최종 회전 행렬 구현 (행렬 곱: @ 연산자)
    rotation_matrix = rotation_matrix_roll @ rotation_matrix_pitch @ rotation_matrix_yaw
    return rotation_matrix


def custom_rotation(pcd: o3d.geometry.PointCloud, rotation_matrix: np.ndarray,
                    center: np.ndarray = np.array([0, 0, 0])) -> o3d.geometry.PointCloud:
    xyz    = np.asarray(pcd.points)
    center = np.asarray(center)

    # center 좌표를 빼서 원점을 center로 이동
    center_translated_xyz = xyz - center
    # 회전 행렬 적용 (Transpose 사용해 (N,3) 형태 유지)
    rotation_xyz = center_translated_xyz @ rotation_matrix.T
    # 회전 후 center 좌표를 다시 더하여 원래 위치로 복원
    rotated_xyz = rotation_xyz + center

    pcd_rotation = o3d.geometry.PointCloud()
    pcd_rotation.points = o3d.utility.Vector3dVector(rotated_xyz)
    return pcd_rotation

### 4. Transformation Matrix

Translation vector와 Rotation matrix를 4×4 행렬 하나로 통합합니다.  
동차 좌표(Homogeneous coordinates)를 사용해 `T @ p_hom` 한 번으로 회전 + 이동을 동시에 처리합니다.

$$T_{4\times4} = \begin{bmatrix} R_{3\times3} & t_{3\times1} \\ 0 & 1 \end{bmatrix}$$

In [ ]:
def custom_get_transformation_matrix(translation_vector: np.ndarray,
                                     rotation_matrix: np.ndarray) -> np.ndarray:
    transformation_matrix = np.eye(4)
    # rotation_matrix를 transformation_matrix에 삽입 (좌상단 3x3)
    transformation_matrix[:3, :3] = rotation_matrix
    # translation_vector를 transformation_matrix에 삽입 (우상단 열)
    transformation_matrix[:3, 3]  = translation_vector
    return transformation_matrix


def custom_pcd_transformation(pcd: o3d.geometry.PointCloud,
                               transformation_matrix: np.ndarray) -> o3d.geometry.PointCloud:
    xyz = np.asarray(pcd.points)
    # (N, 3) -> (N, 4) 동차 좌표로 변환 (마지막 열에 1 추가)
    homogeneous_xyz = np.concatenate([xyz, np.ones((xyz.shape[0], 1))], axis=1)
    # transformation_matrix와 homogeneous_xyz를 곱하여 변환된 좌표 계산
    transformation_xyz = (transformation_matrix @ homogeneous_xyz.T).T
    # 포인트 클라우드를 다시 (N, 3) 형태로 변환
    transformation_xyz = transformation_xyz[:, :3]

    pcd_transformation = o3d.geometry.PointCloud()
    pcd_transformation.points = o3d.utility.Vector3dVector(transformation_xyz)
    return pcd_transformation

### 5. DBSCAN Clustering

밀도 기반 군집화 알고리즘입니다. 파라미터:

- **ε (epsilon)**: 이웃을 정의하는 거리 임계값
- **MinPts**: 코어 포인트가 되기 위한 최소 이웃 개수

흐름:
1. 미분류 점 선택 → `search_radius_vector_3d`로 ε 반경 이웃 탐색
2. 이웃 수 < MinPts → 노이즈 유지 (skip)
3. 이웃 수 ≥ MinPts → 새 클러스터 시작, BFS로 확장
4. 라벨 -1 = 노이즈, 0~ = 클러스터 ID

In [6]:
from typing import Tuple
def custom_dbscan_clustering(pcd: o3d.geometry.PointCloud,
                             epsilon: float,
                             min_points: int) -> Tuple[np.ndarray, int]:
    points   = np.array(pcd.points)
    n_points = len(points)
    kdtree   = o3d.geometry.KDTreeFlann(pcd)

    labels     = np.full(n_points, -1, dtype=int)  # -1: 미분류
    cluster_id = 0

    for point_idx in range(n_points):
        if labels[point_idx] != -1:
            continue

        # kdtree의 search_radius_vector_3d로 epsilon 반경 내 이웃 탐색
        [num_neighbors, neighbors, distances] = kdtree.search_radius_vector_3d(
            points[point_idx], epsilon
        )

        # 이웃 개수가 min_points보다 적으면 노이즈 (라벨 -1 유지)
        if num_neighbors < min_points:
            continue

        # 현재 클러스터 ID로 라벨 업데이트
        labels[point_idx] = cluster_id

        seed_set = list(neighbors)
        i = 0
        while i < len(seed_set):
            current_point = seed_set[i]  # 현재 처리 중인 포인트

            if labels[current_point] == -1:
                # 현재 클러스터 ID로 라벨 업데이트
                labels[current_point] = cluster_id

                # search_radius_vector_3d로 현재 포인트의 이웃 탐색
                [num_neighbors, current_neighbors, distances] = kdtree.search_radius_vector_3d(
                    points[current_point], epsilon
                )
                # 이웃 개수가 min_points 이상이면 코어 포인트 → 이웃 확장
                if len(current_neighbors) > min_points:
                    for neighbor in current_neighbors:
                        if neighbor not in seed_set:
                            # seed_set에 새로운 이웃 추가
                            seed_set.append(neighbor)
            i += 1
        cluster_id += 1

    return labels, cluster_id

## 6주차: Camera & YOLO

Camera 파트는 YOLO(You Only Look Once) 객체 인식 모델을 사용하여 이미지에서 객체를 검출하는 파이프라인을 다룹니다.

전체 흐름은 다음 5단계로 구성됩니다.

- **사전 학습 모델 추론**: `YOLO(model).predict(image)` 로 즉시 객체 검출
- **Dataset 분할**: 이미지를 train / val / test로 분리
- **라벨링**: YOLO 형식(`.txt`) 으로 Bounding Box 라벨 생성 (수동 또는 자동)
- **학습**: `data.yaml` 을 기반으로 커스텀 데이터로 Fine-tuning
- **검증 및 추론**: 학습된 `best.pt` 로 추론 수행 및 결과 시각화


### 0. 환경 설정 및 기본 임포트

- `ultralytics`: YOLO 모델 라이브러리
- `CLASS_NAMES`: 인식할 클래스 목록 (인덱스 순서가 라벨 ID가 됨)
- `YOLO_DATASET_ROOT`: 학습용 데이터셋 폴더 경로


In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path
import random, shutil

YOLO_DATASET_ROOT = Path('./camera_dataset/yolo_tiny')  # YOLO Dataset 저장 경로
SOURCE_IMAGES_DIR = Path('../data/camera/vehicle')     # 원본 이미지 경로
CLASS_NAMES = ['person', 'car', 'bus', 'truck', 'bicycle', 'motorcycle']  # class id = 리스트 인덱스


### 1. 사전 학습 YOLO 모델로 추론

- `YOLO('yolo11n.pt')`: 가중치 파일명으로 모델 로드 (없으면 자동 다운로드)
- `model(image, conf=0.3)`: 신뢰도 0.3 이상인 검출만 반환
- `results[0]`: 이미지 1장일 때 첫 번째 요소가 해당 이미지의 결과
- `result.boxes`: 검출된 Bounding Box 목록
- `result.plot()`: 검출 결과가 그려진 BGR 이미지 반환
- `result.speed['inference']`: 추론 소요 시간 (ms)

**YOLO 모델 크기별 종류** (n → x 순으로 크고 정확하나 느림)
```
yolo11n.pt  # nano  - 가장 빠름
yolo11s.pt  # small
yolo11m.pt  # medium
yolo11l.pt  # large
yolo11x.pt  # xlarge - 가장 정확
```


In [ ]:
model = YOLO('yolo11n.pt')
image = cv2.imread('test.png')
results = model(image, conf=0.3, verbose=True)
result = results[0]

# 추론 결과 출력
inference_ms = result.speed['inference']
num_boxes = len(result.boxes)
detected = sorted(set(result.names[int(b.cls[0])] for b in result.boxes))
print(f'추론 시간: {inference_ms:.1f} ms')
print(f'검출 객체 수: {num_boxes}개')
print(f'검출 클래스: {", ".join(detected)}')

# 결과 시각화
annotated = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(annotated)
plt.axis('off')
plt.title('YOLO 추론 결과')
plt.show()


### 1-1. Bounding Box 정보 추출

- `box.xyxy[0]`: 좌상단/우하단 절대 좌표 `[x1, y1, x2, y2]`
- `box.conf[0]`: 신뢰도(confidence)
- `box.cls[0]`: 클래스 ID (정수)
- `result.names[cls_id]`: 클래스 이름
- YOLO 라벨 포맷: `class_id cx cy w h` (모두 0~1 정규화)


In [ ]:
def boxes_to_df(det_result):
    rows = []
    for box in det_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu().numpy())
        cls_id = int(box.cls[0].cpu().numpy())
        cls_name = det_result.names[cls_id]
        rows.append({'class': cls_name, 'conf': conf,
                     'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                     'w': x2 - x1, 'h': y2 - y1,
                     'cx': (x1 + x2) / 2, 'cy': (y1 + y2) / 2})
    return pd.DataFrame(rows).sort_values('conf', ascending=False).reset_index(drop=True)

df = boxes_to_df(result)
display(df.head(10))


### 2. Dataset 분할 (train / val / test)

- 파일명에 `test`가 포함된 이미지는 자동으로 `images/test`로 분리
- 나머지는 `TRAIN_RATIO` 비율로 train / val 분할
- `RANDOM_SEED`로 재현 가능한 셔플 보장


In [ ]:
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

exts = {'.jpg', '.jpeg', '.png', '.bmp'}
all_images = [p for p in SOURCE_IMAGES_DIR.rglob('*') if p.suffix.lower() in exts]

train_img_dir = YOLO_DATASET_ROOT / 'images' / 'train'
train_lbl_dir = YOLO_DATASET_ROOT / 'labels' / 'train'
val_img_dir   = YOLO_DATASET_ROOT / 'images' / 'val'
val_lbl_dir   = YOLO_DATASET_ROOT / 'labels' / 'val'
test_img_dir  = YOLO_DATASET_ROOT / 'images' / 'test'
for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir, test_img_dir]:
    d.mkdir(parents=True, exist_ok=True)

test_images = [p for p in all_images if 'test' in p.name.lower()]
split_candidates = [p for p in all_images if 'test' not in p.name.lower()]

for src in test_images:
    shutil.copy2(src, test_img_dir / src.name)

random.seed(RANDOM_SEED)
random.shuffle(split_candidates)
n_train = max(1, min(int(len(split_candidates) * TRAIN_RATIO), len(split_candidates) - 1))
for src in split_candidates[:n_train]:
    shutil.copy2(src, train_img_dir / src.name)
for src in split_candidates[n_train:]:
    shutil.copy2(src, val_img_dir / src.name)

n_val = len(split_candidates) - n_train
n_test = len(test_images)
print(f'Train: {n_train} | Val: {n_val} | Test: {n_test}')


### 3. 자동 라벨링 (사전 학습 YOLO 활용)

사전 학습 모델의 예측 결과를 YOLO 라벨 포맷(`.txt`)으로 저장합니다.

**YOLO 라벨 포맷**: `class_id x_center y_center width height` (0~1 정규화)

- COCO class id → 내 데이터셋 class id 로 재매핑 필요 (`name_to_id`)
- 내 클래스에 없는 객체(`cls_name not in name_to_id`)는 건너뜀
- 생성 후 반드시 수동 검수 필요 (오검출/누락 가능)


In [ ]:
pseudo_model = YOLO('yolo11n.pt')
name_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}  # 클래스명 → 내 데이터셋 id

def write_yolo_label_file(txt_path, det_boxes, img_w, img_h, names):
    lines = []
    for box in det_boxes:
        cls_name = names[int(box.cls[0].cpu().numpy())]
        if cls_name not in name_to_id:
            continue
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        # 절대좌표 → 정규화 (cx, cy, w, h)
        x_center = ((x1 + x2) / 2.0) / img_w
        y_center = ((y1 + y2) / 2.0) / img_h
        w = (x2 - x1) / img_w
        h = (y2 - y1) / img_h
        lines.append(f'{name_to_id[cls_name]} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}')
    txt_path.write_text('\n'.join(lines), encoding='utf-8')

for split in ['train', 'val']:
    img_dir = YOLO_DATASET_ROOT / 'images' / split
    lbl_dir = YOLO_DATASET_ROOT / 'labels' / split
    for img_path in sorted(p for p in img_dir.iterdir() if p.suffix.lower() in exts):
        res = pseudo_model(str(img_path), conf=0.35, verbose=False)[0]
        h, w = res.orig_shape
        write_yolo_label_file(lbl_dir / f'{img_path.stem}.txt', res.boxes, w, h, res.names)

print('자동 라벨 생성 완료! 반드시 검수하세요.')


### 4. 모델 학습

#### data.yaml 구조
```yaml
path: /절대경로/yolo_tiny
train: images/train
val: images/val
names:
  0: person
  1: car
  ...
```

- `YOLO('yolo11n.pt').train(data=..., epochs=100, imgsz=640)` 로 Fine-tuning
- 학습 결과는 `runs_camera_yolo/<name>/weights/best.pt` 에 저장
- 데이터가 매우 적으면 과적합 발생 용이 → epochs 조절 또는 augmentation 활용


In [ ]:
# data.yaml 생성
data_yaml_path = YOLO_DATASET_ROOT / 'data.yaml'
data_yaml = f'path: {YOLO_DATASET_ROOT.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnames:\n'
for i, name in enumerate(CLASS_NAMES):
    data_yaml += f'  {i}: {name}\n'
data_yaml_path.write_text(data_yaml, encoding='utf-8')
print('[data.yaml 내용]')
print(data_yaml)

# 학습 수행
train_model = YOLO('yolo11n.pt')  # .pt = 프리트레인, .yaml = 랜덤 초기화
train_results = train_model.train(
    data=str(data_yaml_path),
    epochs=100,
    imgsz=640,
    batch=16,
    project='runs_camera_yolo',
    name='tiny_custom_train',
)
print('학습 완료!')


### 5. 학습된 모델로 검증 및 추론

- `train_results.save_dir`: 학습 결과 저장 디렉토리
- `weights/best.pt`: val loss 기준 최적 가중치
- `result.plot()`: BGR 형식 — matplotlib 표시 전 `cv2.cvtColor(..., BGR2RGB)` 변환 필요


In [ ]:
# best.pt 탐색
run_dir = Path(getattr(train_results, 'save_dir', ''))
candidates = [run_dir / 'weights' / 'best.pt']
candidates += sorted(Path('runs').rglob('best.pt'), key=lambda p: p.stat().st_mtime, reverse=True)
best_pt = next((p for p in candidates if p.exists()), None)

if best_pt:
    trained_model = YOLO(str(best_pt))

    # val 이미지 추론 및 격자 시각화
    val_imgs = sorted(p for p in (YOLO_DATASET_ROOT / 'images' / 'val').glob('*') if p.suffix.lower() in exts)
    val_results = trained_model([str(p) for p in val_imgs], conf=0.25, verbose=False)

    cols = 2
    rows = (len(val_imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
    axes = axes.flatten()
    for i, (path, res) in enumerate(zip(val_imgs, val_results)):
        vis = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
        axes[i].imshow(vis)
        axes[i].set_title(path.name)
        axes[i].axis('off')
    for j in range(len(val_imgs), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('best.pt를 찾지 못했습니다. train 셀을 먼저 실행하세요.')


### 실습: 폴더 내 이미지 일괄 추론

- `model.predict(image_paths, conf=0.25)`: 여러 이미지 한 번에 추론
- `result.path`: 추론에 사용된 이미지 경로
- 결과를 격자(grid) 형태로 시각화


In [ ]:
image_folder = Path('../data/camera/practice').resolve()
image_paths = sorted([*image_folder.glob('*.png'), *image_folder.glob('*.jpg')])
print(f'이미지 수: {len(image_paths)}')

model = YOLO('yolo11n.pt')
results = model.predict(image_paths, conf=0.25)

for res in results:
    n = len(res.boxes) if res.boxes else 0
    print(f'{Path(res.path).name}: {n}개 검출')

# 격자 시각화
cols = 3
rows = (len(results) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten()
for i, res in enumerate(results):
    vis = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    axes[i].imshow(vis)
    axes[i].set_title(Path(res.path).name)
    axes[i].axis('off')
for j in range(len(results), len(axes)):
    axes[j].axis('off')
plt.tight_layout()
plt.show()
